In [8]:
#!/usr/bin/env python3
"""
crawl_demo.py — AI Training Data Crawl Prioritisation Demo
===========================================================

Demonstrates the Quality-Weighted Authority (QWA) heuristic and runs
all 10 experiments on a realistic 42-node synthetic web graph that
includes academic, government, news, tech, and spam pages.

Usage:
    python3.11 crawl_demo.py            # full demo + all experiments
    python3.11 crawl_demo.py --k 15     # change top-k cutoff
    python3.11 crawl_demo.py --out results/crawl
"""
from __future__ import annotations

import argparse
import logging
import sys
import time
from pathlib import Path

import numpy as np
import scipy.sparse as sp

logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s [%(levelname)s] %(name)s: %(message)s",
    datefmt="%H:%M:%S",
    stream=sys.stdout,
)
logger = logging.getLogger("crawl_demo")
import sys
sys.path.insert(0, "../../")

from src.pagerank.core import PageRankEngine
from src.crawler.heuristics import build_all, QualityWeightedAuthority
from src.crawler.experiments import ExperimentSuite
from src.crawler.visualiser import CrawlVisualiser
from src.crawler.quality_proxy import score_url
from src.crawler.prioritizer import CrawlPrioritizer


# ── Synthetic web graph ───────────────────────────────────────────────────────
# 42 URLs with realistic link topology.
# Link patterns:
#   • Wikipedia pages link to primary sources (arXiv, PubMed, .gov)
#   • Academic hubs (arXiv, Stanford, MIT) link to each other heavily
#   • Spam sites link only to each other (sink cluster)
#   • News sites link to government and academic sources
#   • Tech AI labs link to arXiv papers and each other

GRAPH: dict[str, list[str]] = {
    # ── Wikipedia (high PR hub) ──────────────────────────────────────────────
    "https://en.wikipedia.org/wiki/PageRank": [
        "https://arxiv.org/abs/1706.03762",
        "https://en.wikipedia.org/wiki/Graph_theory",
        "https://scholar.google.com/",
        "https://stanford.edu/",
    ],
    "https://en.wikipedia.org/wiki/Graph_theory": [
        "https://en.wikipedia.org/wiki/PageRank",
        "https://acm.org/",
        "https://arxiv.org/abs/2005.14165",
    ],
    "https://en.wikipedia.org/wiki/Machine_learning": [
        "https://arxiv.org/abs/1706.03762",
        "https://deepmind.com/research",
        "https://openai.com/research/gpt-3",
        "https://en.wikipedia.org/wiki/PageRank",
        "https://pubmed.ncbi.nlm.nih.gov/",
    ],
    "https://en.wikipedia.org/wiki/Radioactivity": [
        "https://nih.gov/",
        "https://pubmed.ncbi.nlm.nih.gov/",
        "https://britannica.com/science/radioactivity",
    ],

    # ── arXiv (authoritative academic) ──────────────────────────────────────
    "https://arxiv.org/": [
        "https://arxiv.org/abs/1706.03762",
        "https://arxiv.org/abs/2005.14165",
        "https://scholar.google.com/",
    ],
    "https://arxiv.org/abs/1706.03762": [
        "https://arxiv.org/",
        "https://en.wikipedia.org/wiki/Machine_learning",
        "https://acm.org/",
        "https://stanford.edu/",
    ],
    "https://arxiv.org/abs/2005.14165": [
        "https://arxiv.org/",
        "https://openai.com/research/gpt-3",
        "https://en.wikipedia.org/wiki/Machine_learning",
    ],

    # ── Universities ─────────────────────────────────────────────────────────
    "https://stanford.edu/": [
        "https://arxiv.org/",
        "https://scholar.google.com/",
        "https://openai.com/research/gpt-3",
        "https://mit.edu/",
    ],
    "https://mit.edu/": [
        "https://arxiv.org/",
        "https://scholar.google.com/",
        "https://acm.org/",
        "https://stanford.edu/",
    ],
    "https://berkeley.edu/": [
        "https://arxiv.org/",
        "https://scholar.google.com/",
        "https://mit.edu/",
    ],

    # ── Government / health ──────────────────────────────────────────────────
    "https://nih.gov/": [
        "https://pubmed.ncbi.nlm.nih.gov/",
        "https://cdc.gov/",
        "https://who.int/",
    ],
    "https://pubmed.ncbi.nlm.nih.gov/": [
        "https://nih.gov/",
        "https://arxiv.org/",
        "https://en.wikipedia.org/wiki/Radioactivity",
    ],
    "https://cdc.gov/": [
        "https://nih.gov/",
        "https://who.int/",
    ],
    "https://nasa.gov/": [
        "https://arxiv.org/",
        "https://nih.gov/",
        "https://en.wikipedia.org/wiki/Machine_learning",
    ],
    "https://who.int/": [
        "https://nih.gov/",
        "https://pubmed.ncbi.nlm.nih.gov/",
    ],

    # ── Academic journals ────────────────────────────────────────────────────
    "https://acm.org/": [
        "https://arxiv.org/",
        "https://stanford.edu/",
        "https://scholar.google.com/",
    ],
    "https://scholar.google.com/": [
        "https://arxiv.org/",
        "https://pubmed.ncbi.nlm.nih.gov/",
        "https://acm.org/",
    ],
    "https://nature.com/articles/ai-review": [
        "https://arxiv.org/abs/2005.14165",
        "https://deepmind.com/research",
        "https://nih.gov/",
    ],
    "https://britannica.com/science/radioactivity": [
        "https://en.wikipedia.org/wiki/Radioactivity",
        "https://nih.gov/",
    ],

    # ── AI / Tech ────────────────────────────────────────────────────────────
    "https://openai.com/research/gpt-3": [
        "https://arxiv.org/abs/2005.14165",
        "https://openai.com/",
        "https://stanford.edu/",
    ],
    "https://openai.com/": [
        "https://openai.com/research/gpt-3",
        "https://arxiv.org/",
    ],
    "https://anthropic.com/": [
        "https://arxiv.org/",
        "https://stanford.edu/",
        "https://openai.com/",
    ],
    "https://deepmind.com/research": [
        "https://arxiv.org/abs/1706.03762",
        "https://nature.com/articles/ai-review",
        "https://deepmind.com/",
    ],
    "https://deepmind.com/": [
        "https://deepmind.com/research",
        "https://arxiv.org/",
    ],
    "https://huggingface.co/": [
        "https://arxiv.org/",
        "https://openai.com/",
        "https://anthropic.com/",
    ],

    # ── Quality news ─────────────────────────────────────────────────────────
    "https://reuters.com/technology/ai": [
        "https://openai.com/",
        "https://deepmind.com/",
        "https://en.wikipedia.org/wiki/Machine_learning",
    ],
    "https://bbc.co.uk/news/technology": [
        "https://reuters.com/technology/ai",
        "https://en.wikipedia.org/wiki/Machine_learning",
        "https://who.int/",
    ],
    "https://theguardian.com/technology": [
        "https://bbc.co.uk/news/technology",
        "https://openai.com/",
    ],

    # ── Medium quality ────────────────────────────────────────────────────────
    "https://medium.com/@researcher/attention-is-all-you-need": [
        "https://arxiv.org/abs/1706.03762",
        "https://medium.com/",
    ],
    "https://medium.com/": [
        "https://medium.com/@researcher/attention-is-all-you-need",
    ],
    "https://github.com/huggingface/transformers": [
        "https://huggingface.co/",
        "https://arxiv.org/abs/1706.03762",
    ],
    "https://stackoverflow.com/questions/pagerank": [
        "https://en.wikipedia.org/wiki/PageRank",
        "https://stackoverflow.com/",
    ],
    "https://stackoverflow.com/": [
        "https://stackoverflow.com/questions/pagerank",
    ],

    # ── AI-blocked news sites (robots.txt blocks GPTBot/CCBot/anthropic-ai) ───
    # nytimes.com: verified User-agent: GPTBot / Disallow: /  in their robots.txt
    "https://www.nytimes.com/section/technology": [
        "https://openai.com/research/gpt-3",
        "https://en.wikipedia.org/wiki/Machine_learning",
    ],
    "https://www.cnn.com/tech": [
        "https://www.reuters.com/technology/ai",
        "https://openai.com/",
    ],

    # ── Developer / open-source (robots.txt allows all crawlers) ─────────────
    "https://www.python.org/": [
        "https://github.com/huggingface/transformers",
        "https://stackoverflow.com/questions/pagerank",
        "https://arxiv.org/",
    ],
    "https://docs.python.org/3/": [
        "https://www.python.org/",
        "https://github.com/huggingface/transformers",
    ],
    "https://news.ycombinator.com/": [
        "https://arxiv.org/",
        "https://github.com/huggingface/transformers",
        "https://openai.com/research/gpt-3",
    ],
    "https://substack.com/ai-weekly": [
        "https://openai.com/",
        "https://medium.com/",
    ],
}


def compute_pagerank(graph: dict[str, list[str]], p: float = 0.15) -> dict[str, float]:
    """Compute PageRank on the graph dict."""
    urls = list(graph.keys())
    idx  = {u: i for i, u in enumerate(urls)}
    N    = len(urls)
    out_deg = np.array([len(graph.get(u, [])) for u in urls], dtype=float)
    rows, cols, data = [], [], []
    for src_url, targets in graph.items():
        src = idx[src_url]
        for t in targets:
            if t in idx:
                rows.append(idx[t]); cols.append(src); data.append(1.0)
    if rows:
        A_raw = sp.csr_matrix((data, (rows, cols)), shape=(N, N), dtype=float)
        col_s = np.array(A_raw.sum(axis=0)).flatten()
        col_s[col_s == 0] = 1.0
        A = A_raw.multiply(1.0 / col_s).tocsr()
    else:
        A = sp.eye(N, format="csr") / N
    dangling = out_deg == 0
    result   = PageRankEngine(A, dangling, p=p, tol=1e-10).run()
    return {urls[i]: float(result.scores[i]) for i in range(N)}


def print_top_k_table(ranked, title: str, k: int = 10):
    """Pretty-print top-k crawl candidates."""
    print(f"\n{'='*80}")
    print(f"  {title}")
    print(f"{'='*80}")
    print(f"  {'Rank':>4}  {'Score':>7}  {'URL':<55}  Reason")
    print(f"  {'-'*4}  {'-'*7}  {'-'*55}  {'-'*20}")
    for item in ranked[:k]:
        url_short = item.url[:55] if len(item.url) > 55 else item.url
        print(f"  {item.rank:>4}  {item.score:>7.4f}  {url_short:<55}  {item.reason[:60]}")
    print()


def print_experiment_table(exp_results: dict):
    """Print EXP-10 head-to-head summary table."""
    from src.crawler.heuristics import ALL_HEURISTICS
    short = {
        "H0_Random": "H0 Random",
        "H1_PurePageRank": "H1 Pure PR",
        "H2_HubAuthority": "H2 Hub+PR",
        "H3_PR_Robots": "H3 PR+Robots",
        "H4_QualityWeightedAuth": "H4 QWA ★",
    }
    metrics = ["NDCG@k", "Precision@k", "Recall@k", "Mean Quality", "Unique Domains"]
    print(f"\n{'='*80}")
    print("  EXP-10: Head-to-Head Summary")
    print(f"{'='*80}")
    header = f"  {'Heuristic':<20}" + "".join(f"  {m:>14}" for m in metrics)
    print(header)
    print("  " + "-" * (20 + 16 * len(metrics)))
    for hname, vals in exp_results.items():
        sname = short.get(hname, hname[:20])
        row = f"  {sname:<20}"
        for m in metrics:
            v = vals[m]
            row += f"  {v:>14.4f}" if isinstance(v, float) else f"  {v:>14}"
        print(row)
    print()


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--k",   type=int, default=10)
    parser.add_argument("--p",   type=float, default=0.15)
    parser.add_argument("--out", type=str, default="results/crawl")
    args, _ = parser.parse_known_args()

    out = Path(args.out)
    out.mkdir(parents=True, exist_ok=True)
    vis = CrawlVisualiser(output_dir=str(out))

    # ── Step 1: compute PageRank ─────────────────────────────────────────────
    logger.info("Computing PageRank on %d-node synthetic web graph ...", len(GRAPH))
    pageranks = compute_pagerank(GRAPH, p=args.p)

    # ── Step 2: show all five heuristics ─────────────────────────────────────
    print("\n" + "="*80)
    print("  SYNTHETIC WEB GRAPH — CRAWL PRIORITISATION")
    print(f"  {len(GRAPH)} URLs  ·  p = {args.p}  ·  Top-k = {args.k}")
    print("="*80)

    all_h = build_all(GRAPH, pageranks)
    for h in all_h:
        ranked = h.rank(k=args.k)
        print_top_k_table(ranked, f"{h.name}  —  {h.description}", k=args.k)

    # ── Step 2b: CrawlPrioritizer top-k ──────────────────────────────────────
    prioritizer = CrawlPrioritizer(GRAPH, pageranks)
    print("="*80)
    print("  CRAWL PRIORITIZER (PageRank + Out-Degree frontier queue)")
    print(f"  Policy: {prioritizer.w_pr:.0%} PageRank  +  {prioritizer.w_out:.0%} Out-Degree")
    print("="*80)
    print(prioritizer.explain_policy())
    candidates = prioritizer.top_k(k=args.k)
    print(f"\n  {'Rank':>4}  {'Priority':>8}  {'PageRank':>9}  {'OutDeg':>6}  URL")
    print(f"  {'-'*4}  {'-'*8}  {'-'*9}  {'-'*6}  {'-'*50}")
    for i, c in enumerate(candidates, 1):
        url_short = c.url[:50] if len(c.url) > 50 else c.url
        print(f"  {i:>4}  {c.priority:>8.4f}  {c.pagerank:>9.6f}  {c.out_degree:>6}  {url_short}")
    print()

    # ── Step 3: explain QWA heuristic ────────────────────────────────────────
    print("="*80)
    print("  PROPOSED HEURISTIC: Quality-Weighted Authority (QWA)")
    print("="*80)
    print("""
  Signal Composition
  ──────────────────
  Stage 1 — Hard robots gate: fetch robots.txt and discard domains that block AI crawlers
  Stage 2 — Score permitted URLs:
  score(u) = 0.45 × PageRank_norm(u)       ← link-authority endorsement
           + 0.30 × DomainReputation(u)    ← editorial standards
           + 0.15 × TLDQuality(u)          ← .edu/.gov > .org > .com > .biz
           + 0.10 × URLDepthScore(u)       ← shallow pages = better content

  Why High-PageRank Pages Yield Better AI Training Data
  ──────────────────────────────────────────────────────
  1. FACTUAL ACCURACY  — PageRank is weighted editorial endorsement.
     Many independent trusted authors must have linked to a page, implying
     fact-checking by multiple domain experts.

  2. WRITING QUALITY   — Poorly-written pages lose inbound links over time.
     High-PR pages have survived competitive selection pressure for prose clarity.

  3. STABILITY         — High-PR pages rarely disappear or change URLs.
     Training data from stable sources is less likely to contain broken
     references, hallucination triggers, or stale facts.


  Why QWA Outperforms Pure PageRank
  ───────────────────────────────────
  • Filters spam farms that game PageRank with link schemes
  • Prioritises .edu/.gov where factuality is legally/ethically mandated
  • Robots.txt filter ensures training data is consent-cleared
  • URL depth penalty avoids auto-generated pagination noise
""")

    # ── Step 4: quality signal breakdown for QWA top-10 ─────────────────────
    qwa = QualityWeightedAuthority(GRAPH, pageranks)
    top_k = qwa.rank(k=args.k)
    print("="*80)
    print(f"  QWA Top-{args.k} — Full Signal Breakdown")
    print("="*80)
    print(f"  {'#':>3}  {'PR':>7}  {'Rep':>5}  {'TLD':>5}  {'Dep':>5}  URL")
    print(f"  {'-'*3}  {'-'*7}  {'-'*5}  {'-'*5}  {'-'*5}  {'-'*50}")
    for item in top_k:
        q = score_url(item.url)
        pr = pageranks.get(item.url, 0.0)
        print(
            f"  {item.rank:>3}  {pr:>7.5f}  {q.domain_reputation:>5.2f}  "
            f"{q.tld_score:>5.2f}  {q.url_depth_score:>5.2f}  "
            f"{item.url[:55]}"
        )
    print()

    # ── Step 5: run all 10 experiments ───────────────────────────────────────
    logger.info("Running 10 experiments ...")
    t0    = time.perf_counter()
    suite = ExperimentSuite(GRAPH, pageranks, k=args.k)
    all_r = suite.run_all()
    elapsed = time.perf_counter() - t0
    logger.info("All experiments completed in %.2fs", elapsed)

    # ── Step 6: print summaries ───────────────────────────────────────────────
    r1 = all_r["exp1_ablation"]
    print(f"\n{'='*80}")
    print("  EXP-1: Ablation Study")
    print(f"  Full QWA NDCG@{args.k} = {r1.full_ndcg:.4f}")
    for sig, ndcg in r1.ablations.items():
        drop = r1.full_ndcg - ndcg
        bar  = "▇" * int(drop * 80) if drop > 0 else ""
        print(f"    w/o {sig:<20s}  NDCG={ndcg:.4f}  drop={drop:+.4f}  {bar}")

    r3 = all_r["exp3_correlation"]
    print(f"\n{'='*80}")
    print("  EXP-3: Signal Correlation Summary (most complementary pairs)")
    mat = r3.corr_matrix
    names = r3.signal_names
    pairs = []
    for i in range(len(names)):
        for j in range(i+1, len(names)):
            pairs.append((abs(mat[i,j]), names[i], names[j], mat[i,j]))
    pairs.sort()
    for _, n1, n2, rho in pairs[:3]:
        print(f"    {n1} ↔ {n2}: ρ = {rho:.3f}  (complementary)")

    print_experiment_table(all_r["exp10_head_to_head"])

    # ── Step 7: generate all plots ────────────────────────────────────────────
    logger.info("Generating visualisations ...")
    created = [
        vis.plot_ablation(all_r["exp1_ablation"]),
        vis.plot_p_sensitivity(all_r["exp2_p_sensitivity"]),
        vis.plot_signal_correlation(all_r["exp3_correlation"]),
        vis.plot_domain_diversity(all_r["exp4_diversity"]),
        vis.plot_quality_curve(all_r["exp5_quality_curve"]),
        vis.plot_robots_compliance(all_r["exp6_robots"]),
        vis.plot_url_structural(all_r["exp7_structural"]),
        vis.plot_k_stability(all_r["exp8_k_stability"]),
        vis.plot_topology(all_r["exp9_topology"]),
        vis.plot_head_to_head(all_r["exp10_head_to_head"]),
    ]

    print(f"\n{'='*80}")
    print(f"  Generated {len(created)} figures in {out.resolve()}/")
    for f in created:
        kb = Path(f).stat().st_size / 1024
        print(f"    {Path(f).name:<45s}  {kb:6.1f} KB")
    print(f"{'='*80}\n")


if __name__ == "__main__":
    main()


14:04:12 [INFO] crawl_demo: Computing PageRank on 39-node synthetic web graph ...
14:04:12 [INFO] src.pagerank.core: PageRank converged at iteration 68 (residual=8.553e-11, p=0.150)

  SYNTHETIC WEB GRAPH — CRAWL PRIORITISATION
  39 URLs  ·  p = 0.15  ·  Top-k = 10

  H0_Random  —  Uniform random ordering — control baseline.
  Rank    Score  URL                                                      Reason
  ----  -------  -------------------------------------------------------  --------------------
     1   0.9731  https://www.nytimes.com/section/technology               random baseline
     2   0.9572  https://huggingface.co/                                  random baseline
     3   0.8922  https://arxiv.org/abs/2005.14165                         random baseline
     4   0.8617  https://substack.com/ai-weekly                           random baseline
     5   0.8475  https://medium.com/@researcher/attention-is-all-you-nee  random baseline
     6   0.8294  https://docs.python.org/3/    

/Users/alexshienhowkhoo/Documents/NTU_BCG/NTU BCG Y4S2/PageRank_AI_Graph_Rag/src/crawler/../../src/crawler/visualiser.py:146: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax.set_xticklabels(drop_labels, rotation=20, ha="right", fontsize=8)


14:04:16 [INFO] src.crawler.visualiser: Saved: results/crawl/crawl_exp1_ablation.png
14:04:19 [INFO] src.crawler.visualiser: Saved: results/crawl/crawl_exp2_p_sensitivity.png
14:04:21 [INFO] src.crawler.visualiser: Saved: results/crawl/crawl_exp3_signal_correlation.png
14:04:24 [INFO] src.crawler.visualiser: Saved: results/crawl/crawl_exp4_domain_diversity.png
14:04:25 [INFO] src.crawler.visualiser: Saved: results/crawl/crawl_exp5_quality_curve.png
14:04:27 [INFO] src.crawler.visualiser: Saved: results/crawl/crawl_exp6_robots_compliance.png
14:04:29 [INFO] src.crawler.visualiser: Saved: results/crawl/crawl_exp7_structural.png
14:04:31 [INFO] src.crawler.visualiser: Saved: results/crawl/crawl_exp8_k_stability.png
14:04:33 [INFO] src.crawler.visualiser: Saved: results/crawl/crawl_exp9_topology.png
14:04:39 [INFO] src.crawler.visualiser: Saved: results/crawl/crawl_exp10_head_to_head.png

  Generated 10 figures in /Users/alexshienhowkhoo/Documents/NTU_BCG/NTU BCG Y4S2/PageRank_AI_Graph_Rag

In [9]:
parser = argparse.ArgumentParser()
parser.add_argument("--k",   type=int, default=10)
parser.add_argument("--p",   type=float, default=0.15)
parser.add_argument("--out", type=str, default="results/crawl")
args, _ = parser.parse_known_args()

pageranks = compute_pagerank(GRAPH, p=args.p)
pageranks

14:04:40 [INFO] src.pagerank.core: PageRank converged at iteration 68 (residual=8.553e-11, p=0.150)


{'https://en.wikipedia.org/wiki/PageRank': 0.018871723889584198,
 'https://en.wikipedia.org/wiki/Graph_theory': 0.007856395172842992,
 'https://en.wikipedia.org/wiki/Machine_learning': 0.0413650244313652,
 'https://en.wikipedia.org/wiki/Radioactivity': 0.029870550921662464,
 'https://arxiv.org/': 0.13214313786500576,
 'https://arxiv.org/abs/1706.03762': 0.06649244239929052,
 'https://arxiv.org/abs/2005.14165': 0.06205501912755019,
 'https://stanford.edu/': 0.05598458992018618,
 'https://mit.edu/': 0.016832622795000883,
 'https://berkeley.edu/': 0.003846153846153846,
 'https://nih.gov/': 0.06577602520933436,
 'https://pubmed.ncbi.nlm.nih.gov/': 0.07338659831014958,
 'https://cdc.gov/': 0.022482694318192806,
 'https://nasa.gov/': 0.003846153846153846,
 'https://who.int/': 0.033590724016540084,
 'https://acm.org/': 0.044911178208198684,
 'https://scholar.google.com/': 0.07458518602285102,
 'https://nature.com/articles/ai-review': 0.008898371123009619,
 'https://britannica.com/science/radi

In [10]:
all_h = build_all(GRAPH, pageranks)
all_h

In [14]:
# Import fetch_robots_permitted from the module (do not redefine inline)
import importlib
import src.crawler.prioritizer as _prio_mod
importlib.reload(_prio_mod)  # pick up any edits to prioritizer.py without kernel restart
from src.crawler.prioritizer import fetch_robots_permitted, _parse_robots_content

In [15]:
fetch_robots_permitted("https://www.python.org/")

True

In [16]:
# Robots.txt verification — expected results confirmed by live fetch
# BLOCKED: explicit AI bot entries with Disallow: /
print("--- nytimes.com (BLOCKED) ---")
print(fetch_robots_permitted("https://www.nytimes.com/section/technology"))   # expect False

print("\n--- cnn.com (BLOCKED) ---")
print(fetch_robots_permitted("https://www.cnn.com/tech"))                     # expect False

print("\n--- washingtonpost.com (BLOCKED) ---")
print(fetch_robots_permitted("https://www.washingtonpost.com/technology"))    # expect False

# ALLOWED: open robots.txt, no AI bot blocks
print("\n--- python.org (ALLOWED) ---")
print(fetch_robots_permitted("https://www.python.org/"))                      # expect True

--- nytimes.com (BLOCKED) ---
False

--- cnn.com (BLOCKED) ---
False

--- washingtonpost.com (BLOCKED) ---
False

--- python.org (ALLOWED) ---
True


# Archive

In [7]:
# test_prioritizer.py  (run from project root: python test_prioritizer.py)

import sys
sys.path.insert(0, "src")

# ── 1. Toy graph ────────────────────────────────────────────────────────────
# 5 nodes, adjacency dict: {url: [outlinks]}
graph = {
            "https://arxiv.org/": ["https://arxiv.org/paper1", "https://arxiv.org/paper2"],
            "https://arxiv.org/paper1": ["https://arxiv.org/"],
            "https://arxiv.org/paper2": [],
            "https://spam.com/": [],
    }
# ── 2. Fake precomputed PageRank scores ─────────────────────────────────────
pageranks = {
            "https://arxiv.org/": 0.5,
            "https://arxiv.org/paper1": 0.2,
            "https://arxiv.org/paper2": 0.2,
            "https://spam.com/": 0.01,
        }

# ── 3. Run prioritizer ──────────────────────────────────────────────────────
p = CrawlPrioritizer(
    graph=graph,
    pageranks=pageranks,
    w_pr=0.6,       # 60% PageRank
    w_out=0.2,      # 20% out-degree
    w_robots=0.2,   # 20% robots (check_robots=False → all get 1.0)
    check_robots=True,
)


TypeError: CrawlPrioritizer.__init__() got an unexpected keyword argument 'w_robots'

In [ ]:

# ── 4. Print top-k results ──────────────────────────────────────────────────
print("=== Top 5 URLs to crawl ===\n")
for rank, candidate in enumerate(p.top_k(k=5), 1):
    print(f"#{rank}  {candidate.url}")
    print(f"     priority  : {candidate.priority:.4f}")
    print(f"     pagerank  : {candidate.pagerank:.4f}")
    print(f"     out_degree: {candidate.out_degree}")
    print(f"     robots_ok : {candidate.robots_ok}")
    print(f"     reason    : {candidate.reason}")
    print()



=== Top 5 URLs to crawl ===

pr_array [0.5  0.2  0.2  0.01]
ord_array [2. 1. 0. 0.]
pr_norm [1.        0.3877551 0.3877551 0.       ]
od_norm [1.  0.5 0.  0. ]
Robots_Array [1. 1. 1. 1.]
Priority_Array [1.         0.53265306 0.43265306 0.2       ]
#1  https://arxiv.org/
     priority  : 1.0000
     pagerank  : 0.5000
     out_degree: 2
     robots_ok : True
     reason    : very high PageRank authority; high out-degree hub; robots.txt allows AI crawlers

#2  https://arxiv.org/paper1
     priority  : 0.5327
     pagerank  : 0.2000
     out_degree: 1
     robots_ok : True
     reason    : robots.txt allows AI crawlers

#3  https://arxiv.org/paper2
     priority  : 0.4327
     pagerank  : 0.2000
     out_degree: 0
     robots_ok : True
     reason    : robots.txt allows AI crawlers

#4  https://spam.com/
     priority  : 0.2000
     pagerank  : 0.0100
     out_degree: 0
     robots_ok : True
     reason    : robots.txt allows AI crawlers



In [ ]:
# ── 5. Print crawl policy explanation ───────────────────────────────────────
print(p.explain_policy())


Crawl Priority Policy
  PageRank weight  : 60%  — authoritative pages first
  Out-degree weight: 20%  — hubs surface more new URLs
  Robots bonus     : 20%  — prefer consent-cleared pages

Why high-PageRank pages yield better AI training data:
  PageRank is a proxy for editorial endorsement — many trusted
  sites must have found a page worth linking to.  High-PR pages
  tend to be factually accurate, well-written, and frequently
  updated (e.g. Wikipedia, arXiv, official documentation).
  These properties directly correlate with training data quality
  for generative models.

Robots.txt heuristic:
  Pages whose domain does NOT block known AI bots (GPTBot,
  CCBot, anthropic-ai) in robots.txt are more likely to have
  consented to AI training use.  Combining PageRank authority
  with robots-crawlability surfaces high-quality, consent-cleared
  training pages.


In [ ]:
def _is_crawlable(url: str) -> bool:
    """Check robots.txt for the domain. Cached per domain."""
    try:
        parsed = urllib.parse.urlparse(url)
        print(parsed)
        domain = f"{parsed.scheme}://{parsed.netloc}"
        print(domain)
    except Exception:
        return True  # assume crawlable on parse error


    # robots_url = f"{domain}/robots.txt"
    # try:
    #     import urllib.request
    #     with urllib.request.urlopen(robots_url, timeout=5) as resp:
    #         content = resp.read().decode("utf-8", errors="replace")
    # except Exception:
    #     result = True  # network error → assume crawlable

    # return result

_is_crawlable("https://arxiv.org/")

ParseResult(scheme='https', netloc='arxiv.org', path='/', params='', query='', fragment='')
https://arxiv.org


In [ ]:
url = "https://arxiv.org/"
parsed = urllib.parse.urlparse(url)
print(parsed)

ParseResult(scheme='https', netloc='arxiv.org', path='/', params='', query='', fragment='')


In [ ]:
domain = f"{parsed.scheme}://{parsed.netloc}"
domain

'https://arxiv.org'

In [ ]:
robots_url = f"{domain}/robots.txt"
import urllib.request
with urllib.request.urlopen(robots_url, timeout=5) as resp:
    content = resp.read().decode("utf-8", errors="replace")
content

'# robots.txt for http://arxiv.org/ and mirror sites http://*.arxiv.org/\n# Indiscriminate automated downloads from this site are not permitted\n# See also: http://arxiv.org/help/robots\n\nUser-agent: *\nCrawl-delay: 15\nAllow: /archive\nAllow: /year\nAllow: /list\nAllow: /abs\nAllow: /pdf\nAllow: /html\nAllow: /catchup\nDisallow: /user\nDisallow: /e-print\nDisallow: /src\nDisallow: /ps\nDisallow: /dvi\nDisallow: /cookies\nDisallow: /form\nDisallow: /find\nDisallow: /view\nDisallow: /ftp\nDisallow: /refs\nDisallow: /cits\nDisallow: /format\nDisallow: /PS_cache\nDisallow: /Stats\nDisallow: /seek-and-destroy\nDisallow: /IgnoreMe\nDisallow: /oai2\nDisallow: /auth\nDisallow: /tb\nDisallow: /tb-recent\nDisallow: /trackback\nDisallow: /prevnext\nDisallow: /ct\nDisallow: /api\nDisallow: /search\nDisallow: /set_author_id\nDisallow: /show-email\n\nUser-agent: Googlebot\nAllow: /archive\nAllow: /year\nAllow: /list\nAllow: /abs\nAllow: /pdf\nAllow: /html\nAllow: /catchup\nDisallow: /user\nDisallo

In [ ]:
def _parse_robots(content: str) -> bool:
    """
    Return True if none of the known AI bots are globally disallowed.

    Simple parser: looks for User-agent: <bot> followed by Disallow: /
    """
    lines = content.splitlines()
    current_agents: list[str] = []
    for line in lines:
        line = line.strip()
        ua_match = _USERAGENT_RE.match(line)
        if ua_match:
            print(line)
            print(f"ua_match:{ua_match}")
            
            current_agents = [ua_match.group(1).strip()]
            continue
        dis_match = _ROBOTS_DISALLOW_RE.match(line)
        if dis_match:
            print(line)
            print(f"dis_match: {dis_match}")
            path = dis_match.group(1).strip()
            if path in ("/", "*"):
                for agent in current_agents:
                    for bot in _AI_BOTS:
                        if bot.lower() in agent.lower() or agent == "*":
                            return False  # globally disallowed
    return True

_parse_robots(content)

User-agent: *
ua_match:<re.Match object; span=(0, 13), match='User-agent: *'>
Disallow: /user
dis_match: <re.Match object; span=(0, 15), match='Disallow: /user'>
Disallow: /e-print
dis_match: <re.Match object; span=(0, 18), match='Disallow: /e-print'>
Disallow: /src
dis_match: <re.Match object; span=(0, 14), match='Disallow: /src'>
Disallow: /ps
dis_match: <re.Match object; span=(0, 13), match='Disallow: /ps'>
Disallow: /dvi
dis_match: <re.Match object; span=(0, 14), match='Disallow: /dvi'>
Disallow: /cookies
dis_match: <re.Match object; span=(0, 18), match='Disallow: /cookies'>
Disallow: /form
dis_match: <re.Match object; span=(0, 15), match='Disallow: /form'>
Disallow: /find
dis_match: <re.Match object; span=(0, 15), match='Disallow: /find'>
Disallow: /view
dis_match: <re.Match object; span=(0, 15), match='Disallow: /view'>
Disallow: /ftp
dis_match: <re.Match object; span=(0, 14), match='Disallow: /ftp'>
Disallow: /refs
dis_match: <re.Match object; span=(0, 15), match='Disallow: /ref

True

In [ ]:
from collections import deque
import urllib.request
from html.parser import HTMLParser

class LinkParser(HTMLParser):
    """Extract all href links from a page."""
    def __init__(self):
        super().__init__()
        self.links = []

    def handle_starttag(self, tag, attrs):
        if tag == "a":
            for attr, val in attrs:
                if attr == "href" and val:
                    self.links.append(val)


def crawl(seed_url, max_pages=10):
    visited = set()
    queue = deque([seed_url])

    while queue and len(visited) < max_pages:

        # ── 1. Pick next URL ──────────────────────────────────
        url = queue.popleft()
        if url in visited:
            continue

        # ── 2. Fetch page ─────────────────────────────────────
        try:
            with urllib.request.urlopen(url, timeout=5) as resp:
                html = resp.read().decode("utf-8", errors="replace")
                print(html)
        except Exception as e:
            print(f"  FAILED: {url} — {e}")
            continue

        # ── 3. Mark visited ───────────────────────────────────
        visited.add(url)
        print(f"  Crawled: {url}")

        # ── 4. Extract links ──────────────────────────────────
        parser = LinkParser()
        parser.feed(html)
        
        for link in parser.links:
            full_url = urllib.parse.urljoin(url, link) 
            print(f"Full url {full_url}")
            if full_url.startswith("http") and link not in visited:
                queue.append(full_url)

    return visited


# Run
crawl("http://quotes.toscrape.com", max_pages=5)


<!DOCTYPE html>
<html lang="en">
<head>
	<meta charset="UTF-8">
	<title>Quotes to Scrape</title>
    <link rel="stylesheet" href="/static/bootstrap.min.css">
    <link rel="stylesheet" href="/static/main.css">
    
    
</head>
<body>
    <div class="container">
        <div class="row header-box">
            <div class="col-md-8">
                <h1>
                    <a href="/" style="text-decoration: none">Quotes to Scrape</a>
                </h1>
            </div>
            <div class="col-md-4">
                <p>
                
                    <a href="/login">Login</a>
                
                </p>
            </div>
        </div>
    

<div class="row">
    <div class="col-md-8">

    <div class="quote" itemscope itemtype="http://schema.org/CreativeWork">
        <span class="text" itemprop="text">“The world as we have created it is a process of our thinking. It cannot be changed without changing our thinking.”</span>
        <span>by <small class="auth

{'http://quotes.toscrape.com',
 'http://quotes.toscrape.com/',
 'http://quotes.toscrape.com/author/Albert-Einstein',
 'http://quotes.toscrape.com/login',
 'http://quotes.toscrape.com/tag/change/page/1/'}